# ROGII - diagnose the shared "hard well" 86454a6f

Flagged in the competitor-ideas backlog: top competitors (Shrey #4, Rishikesh #3, us) all score ~40-46 on this ONE well, while several lower-ranked competitors score 19-25 on it -- suggesting real unexploited headroom, but nobody described the mechanism. Never diagnosed before.

Run: Input = competition dataset. CPU. Internet off. Run All.


In [ ]:
"""Diagnose well 86454a6f: flagged in the competitor-ideas backlog as a shared pain point across TOP
competitors (Shrey #4: 43.86, Rishikesh #3: 40.8, us: 43.5 via full_combo) while several LOWER-ranked
competitors do much better on this SPECIFIC well (k256.dev #12: 20.76, Connor #1314: 19, Andrey #423:
23-24.61, Sandy: 22.64) -- suggesting real unexploited headroom, but no competitor ever described the
actual mechanism. Never given its own dedicated diagnostic in this project. Goal: characterize what's
structurally different/hard about this well (trajectory shape, GR quality, typewell match quality,
known-zone dip stability, TVT drift magnitude/direction) to see if a targeted (not general) fix is
findable, and whether it's a wall-mechanism we've already mapped (broadband GR self-similarity, unstable
dip) or something genuinely new.
"""
import glob, os, numpy as np, pandas as pd

_c = glob.glob('/kaggle/input/**/*__horizontal_well.csv', recursive=True)
_t = [p for p in _c if 'train' in p.lower()]
_c = _t if _t else _c
TRAIN_DIR = os.path.dirname(_c[0]) if _c else 'd:/ROGII/data/train'
print(f'TRAIN_DIR={TRAIN_DIR}', flush=True)

WID = '86454a6f'
hw = pd.read_csv(f'{TRAIN_DIR}/{WID}__horizontal_well.csv')
tw = pd.read_csv(f'{TRAIN_DIR}/{WID}__typewell.csv').sort_values('TVT')
tw_tvt = tw['TVT'].values.astype(float)
tw_gr = tw['GR'].fillna(tw['GR'].mean()).values.astype(float)

km = hw['TVT_input'].notna()
n_known = int(km.sum()); n_eval = int((~km).sum())
print(f'n_known={n_known}  n_eval={n_eval}  n_total={len(hw)}', flush=True)

MD = hw['MD'].values.astype(float)
Z = hw['Z'].values.astype(float)
GR = hw['GR'].values.astype(float)
TVT_input = hw['TVT_input'].values.astype(float)
TVT_true = hw['TVT'].values.astype(float) if 'TVT' in hw.columns else None

known_idx = np.flatnonzero(km.values)
eval_idx = np.flatnonzero(~km.values)
last_known = known_idx[-1]

# 1) known-zone dip stability: local dip over sliding windows in the known zone
kn_md = MD[known_idx]; kn_tvt = TVT_input[known_idx]
dip_local = np.gradient(kn_tvt) / np.maximum(np.gradient(kn_md), 1e-6)
print(f'known-zone dip: median={np.median(dip_local):.4f} std={np.std(dip_local):.4f} '
      f'min={dip_local.min():.4f} max={dip_local.max():.4f}', flush=True)
tail = pd.DataFrame({'MD': kn_md, 'TVT': kn_tvt}).tail(30)
dt = np.diff(tail['TVT'].values); dm = np.diff(tail['MD'].values)
m = dm > 0
tail_dip = float(np.median(dt[m] / dm[m])) if m.sum() >= 3 else float('nan')
print(f'tail-30 dip estimate (what a tracker would extrapolate from): {tail_dip:.4f}', flush=True)

# 2) eval-zone TRUE drift (only visible because this is a TRAIN well) -- magnitude, direction changes
if TVT_true is not None:
    ev_md = MD[eval_idx]; ev_true = TVT_true[eval_idx]
    ev_dip = np.gradient(ev_true) / np.maximum(np.gradient(ev_md), 1e-6)
    total_drift = ev_true[-1] - TVT_input[last_known]
    max_swing = ev_true.max() - ev_true.min()
    n_sign_changes = int(np.sum(np.diff(np.sign(pd.Series(ev_dip).rolling(21, center=True, min_periods=5).mean().fillna(0).values)) != 0))
    print(f'eval-zone true TVT: total_drift={total_drift:.2f}  max_swing={max_swing:.2f}  '
          f'dip sign-changes(smoothed)={n_sign_changes}', flush=True)
    print(f'eval-zone true dip: median={np.median(ev_dip):.4f} std={np.std(ev_dip):.4f}', flush=True)
    print(f'known-tail-dip vs eval-median-dip gap: {tail_dip - np.median(ev_dip):.4f} '
          f'(large gap = tail extrapolation would badly mislead a tracker)', flush=True)

# 3) GR quality / typewell match quality in known zone (calibration residual)
kn_gr = GR[known_idx]
twk = np.interp(kn_tvt, tw_tvt, tw_gr)
v = np.isfinite(kn_gr) & np.isfinite(twk)
a, b = np.polyfit(kn_gr[v], twk[v], 1) if v.sum() >= 20 else (1., 0.)
cal = kn_gr * a + b
resid = cal[v] - twk[v]
print(f'known-zone GR-vs-typewell calib residual std: {np.std(resid):.3f}  (gs_sigma equivalent)', flush=True)
print(f'known-zone GR-typewell correlation (post-cal): {np.corrcoef(cal[v], twk[v])[0,1]:.3f}', flush=True)

# 4) typewell self-similarity: how "generic"/repetitive is the GR curve
tw_autocorr = np.correlate(tw_gr - tw_gr.mean(), tw_gr - tw_gr.mean(), mode='full')
tw_autocorr = tw_autocorr[len(tw_autocorr)//2:] / tw_autocorr[len(tw_autocorr)//2]
half_life_idx = np.argmax(tw_autocorr < 0.5) if (tw_autocorr < 0.5).any() else -1
print(f'typewell length={len(tw_tvt)}  GR autocorr half-life (samples): {half_life_idx}', flush=True)

# 5) geometry: XY drift, azimuth stability (does the well turn/twist a lot?)
X = hw['X'].values.astype(float); Y = hw['Y'].values.astype(float)
head = np.arctan2(np.gradient(Y), np.gradient(X) + 1e-9)
head_deg = np.degrees(head)
head_change = np.abs(np.diff(np.unwrap(head)))
print(f'azimuth total absolute turn (deg): {np.degrees(head_change.sum()):.1f}  '
      f'max single-step turn (deg): {np.degrees(head_change.max()):.2f}', flush=True)

print('\n--- summary for comparison against other wells (run a small sample too) ---', flush=True)
# quick comparison sample: 15 other random wells' same stats for context
rng = np.random.default_rng(0)
all_wids = sorted({os.path.basename(f).split('__')[0] for f in glob.glob(f'{TRAIN_DIR}/*__horizontal_well.csv')})
sample = [w for w in rng.choice(all_wids, size=40, replace=False) if w != WID][:15]
rows = []
for w in sample:
    try:
        _hw = pd.read_csv(f'{TRAIN_DIR}/{w}__horizontal_well.csv')
        if 'TVT' not in _hw.columns: continue
        _km = _hw['TVT_input'].notna()
        if _km.sum() < 20 or (~_km).sum() < 20: continue
        _ev_true = _hw['TVT'].values.astype(float)[~_km.values]
        _swing = _ev_true.max() - _ev_true.min()
        _drift = _ev_true[-1] - _hw['TVT_input'].values.astype(float)[np.flatnonzero(_km.values)[-1]]
        rows.append((w, _swing, _drift))
    except Exception:
        continue
swings = [r[1] for r in rows]; drifts = [abs(r[2]) for r in rows]
print(f'sample(n={len(rows)}) eval max_swing: median={np.median(swings):.1f} vs 86454a6f={max_swing:.1f}', flush=True)
print(f'sample(n={len(rows)}) |eval total drift|: median={np.median(drifts):.1f} vs 86454a6f={abs(total_drift):.1f}', flush=True)

